## Streamlined Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
from shapely.geometry import Polygon
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set a bbox for investigation
bbox = box(-117.161615,32.703371,-117.129171,32.720505)

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
   "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [4]:
# import Zillow data (make take 10-20 minutes)
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint
There are a total of 8 parquet files that correspond to California. We select the one that contain Los Angeles county (w120_n35_w115_n30.parquet).

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Streamlined Code (takes 1 minute!)

In [ ]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to multi-family home residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

In [9]:
parcels_res

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry
23,4241720700,San Diego,None,None,None,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,..."
118,4241721300,San Diego,None,None,None,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,..."
119,4241722000,San Diego,None,None,None,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,..."
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,..."
154,4236220200,San Diego,None,None,None,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,..."
...,...,...,...,...,...,...,...,...
1039997,2581831500,San Diego,None,None,None,91.675357,462.777931,"MULTIPOLYGON Z (((-117.29489 33.04032 0.00000,..."
1040095,5182013023,San Diego,None,None,None,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,..."
1040096,5182013025,San Diego,None,None,None,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,..."
1040121,2181606400,San Diego,None,None,None,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,..."


In [ ]:
fig, ax = plt.subplots()
parcels_res.plot(ax = ax,
                 color = 'red')

Change the code to keep the `PARNO` (parcel number with each of the buildings).

In [10]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects")

# subset for the columns we need 
valid_buildings = valid_buildings[['source', 'id', 'height', 'var', 'bbox', 'geometry', 'PARNO']]

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)


In [11]:
valid_buildings

,source,id,height,var,bbox,geometry,PARNO
6344544,osm,546629542,2.095237,0.773937,"{'xmax': -116.79879449999999, 'xmin': -116.799...","POLYGON ((-116.79891 33.39104, -116.79901 33.3...",1131300900
6344637,osm,546630246,2.132655,0.123639,"{'xmax': -116.7952663, 'xmin': -116.7954647999...","POLYGON ((-116.79539 33.39013, -116.79546 33.3...",1131300900
6344638,ms,UnitedStates_023013203_218928,8.093865,2.164009,"{'xmax': -116.79542502150233, 'xmin': -116.795...","POLYGON ((-116.79555 33.39021, -116.79543 33.3...",1131300900
6344639,ms,UnitedStates_023013203_277953,6.538584,2.185169,"{'xmax': -116.7951542354982, 'xmin': -116.7952...","POLYGON ((-116.79529 33.38950, -116.79524 33.3...",1131300900
6344640,ms,UnitedStates_023013203_277952,6.395244,3.757385,"{'xmax': -116.79521751197618, 'xmin': -116.795...","POLYGON ((-116.79522 33.38880, -116.79522 33.3...",1131300900
...,...,...,...,...,...,...,...
7720646,ms,UnitedStates_023013221_617530,3.968409,0.611403,"{'xmax': -116.76287700700234, 'xmin': -116.763...","POLYGON ((-116.76306 32.62119, -116.76305 32.6...",6491600700
7720648,ms,UnitedStates_023013221_20839,4.138736,1.040947,"{'xmax': -116.76272701637753, 'xmin': -116.762...","POLYGON ((-116.76281 32.62073, -116.76281 32.6...",6491600700
7720672,ms,UnitedStates_023013221_415526,1.914224,0.045000,"{'xmax': -116.7523910988333, 'xmin': -116.7525...","POLYGON ((-116.75253 32.61698, -116.75239 32.6...",6491810300
7720673,ms,UnitedStates_023013221_118474,1.503267,0.086076,"{'xmax': -116.75161930609684, 'xmin': -116.751...","POLYGON ((-116.75162 32.61623, -116.75169 32.6...",6491810300


In [ ]:
## CREATE CENTROIDS FOR EACH MULTI-RESIDENTIAL BUILDINGS
# change the crs to a projected crs as centroids only works in projected crs
valid_buildings = valid_buildings.to_crs('EPSG:3857')

# create centroids for each multi residential builing
valid_building_centroids = valid_buildings.centroid

## join the centroids with the valid buildings to keep the additional building information 
# select only the necessary columns of the dataframe
valid_building_ids = valid_buildings[['source', 'id', 'height', 'var', 'PARNO']]

# make the centroids a dataframe for the join
valid_building_centroids = pd.DataFrame(valid_building_centroids)

# join the centroids with the building ids
multi_buildings = valid_building_ids.join(valid_building_centroids)


In [ ]:
## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)


# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# reproject data frame to crs with meters as units
building_m = building_zillow.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# 6. keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

multi_complete = multi_complete.drop(['index_right'], axis = 1)

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# join back to parcels
units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# join on parcels with summed number of units
multi_summed_units = parcels_res.join(units_multi)

assert len(multi_summed_units) < len(multi_by_parcel)

# save all non-multi observations
non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)

# keep only variables of interest
non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

## Final results

In [ ]:
multi_complete.head() # should be my buildings??

In [ ]:
multi_summed_units.head(3)

In [ ]:
multi_by_parcel.head(3)

In [ ]:
non_multi_points.head(3)

## Renaming and saving

In [ ]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [ ]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

In [ ]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [ ]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [ ]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')